In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql import functions as F

In [0]:
df =spark.read.table("samples.bakehouse.sales_transactions")
display(df)

In [0]:
#first record
display(df.first())
#last record
#display(df.last())
display("""""""""""""""""""""""""""""""""""""""""")
#df.orderBy(col("datetime").desc()).limit(1).display()
df.select("*") \
  .orderBy(expr("datetime DESC")).limit(1) \
  .display()

In [0]:
%sql
select min(quantity),max(quantity) from samples.bakehouse.sales_transactions

In [0]:
df2=df.select(min("quantity").alias("min_quant"),max("quantity").alias("max_quant"))
display(df2)

In [0]:
df1=df.groupBy("product").count().filter(col("count")>500)
display(df1)

In [0]:
%sql
select product,count(product) as count
from samples.bakehouse.sales_transactions
group by product
having count(product) > 500
order by count

In [0]:
from pyspark.sql.functions import collect_set, collect_list
df2=df.groupBy("paymentMethod").agg(collect_set("product"), collect_list("product"))
display(df2)

In [0]:
%sql
select paymentMethod,collect_set(product),collect_list(product)
from samples.bakehouse.sales_transactions
group by paymentMethod


In [0]:
window_spec = Window.partitionBy("Product").orderBy(col("quantity").desc())
df1 = df.withColumn("row_number_over_product",row_number().over(window_spec))
df2 =df1.withColumn("rank_number_over_product",rank().over(window_spec))
df3= df2.withColumn("dense_rank_over_product",dense_rank().over(window_spec))
df4=df3.select("transactionID","customerID","product","quantity","row_number_over_product","rank_number_over_product","dense_rank_over_product")
display(df4)


In [0]:
df3.printSchema()

In [0]:
%sql
select transactionId,customerId,product,row_number() over(partition by product order by quantity desc) as row_number_over_product,
rank() over(partition by product order by quantity desc) as rank_over_product,
dense_rank() over(partition by product order by quantity desc) as dense_rank_over_product
from samples.bakehouse.sales_transactions